In [62]:
import time
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import undetected_chromedriver as uc
import random
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC

In [59]:
# using uc because of captcha issue
driver = uc.Chrome(version_main=146) # my webdriver uses version 146 and this uc usig 147 so to make uc on same level i did this
driver.maximize_window()

# explicit wait
wait = WebDriverWait(driver,10)


def wait_for_page_to_load(driver,wait):
    page_title = driver.title
    try:
        wait.until(
            lambda driver: driver.execute_script("return document.readyState") == 'complete'
        )
    except:
        print(f"The page \"{page_title}\" could not load sucessfully! maybe session timed out")
    else:
         print(f"The page \"{page_title}\" loaded sucessfully!")

url = 'https://finance.yahoo.com/'
driver.get(url)
wait_for_page_to_load(driver,wait)

# hovering over maarket menu
actions = ActionChains(driver)

# before we hover on the market element we will check the precense of the element through excplicit wait and EC
market_menu = wait.until(
    EC.visibility_of_element_located((By.XPATH,"//div[normalize-space()='Markets']")) # why tuple?
)
actions.move_to_element(market_menu).perform() # but market_menu gives boolean then how did it knew the x path of market element

# click on stocks
stocks_element = wait.until(
    EC.visibility_of_element_located((By.XPATH,"//a[@href='https://finance.yahoo.com/markets/stocks/']//span[@class='item-content yf-ngf7r6'][normalize-space()='Stocks']"))
)
actions.move_to_element(stocks_element).perform()

active_element = wait.until(
    EC.visibility_of_element_located((By.XPATH,"/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[1]/a[1]/span[1]")) # how is this function different from presence wala method
)

active_element.click()
wait_for_page_to_load(driver,wait)


data =[]
# scraping the data
while True:
    try:
        
        #scraping
        wait.until(
            EC.presence_of_element_located((By.TAG_NAME,'table'))
        )
    
        rows = driver.find_elements(By.CSS_SELECTOR,"table tbody tr")
    
        for row in rows:
            values = row.find_elements(By.TAG_NAME,"td")
                
            stock = {
            'name': values[1].text,
            'symbol':values[0].text,
            'price':values[3].text,
            'change':values[4].text,
            'volume':values[6].text,
            'market_cap':values[8].text,
            'pe_ratio':values[9].text,
        }
            data.append(stock)
    except: 
        print('All data collected!')

    
    # click next
    try:
        next_button = wait.until(
            EC.element_to_be_clickable((By.XPATH,'//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]'))
        )

    except:
        print("The next button is not clickable. We have navigated through all the pages!")
        break
    else:
        next_button.click()
        # time.sleep(1)

driver.quit()

The page "Yahoo Finance - Stock Market Live, Quotes, Business & Finance News" loaded sucessfully!
The page "Yahoo Finance - Stock Market Live, Quotes, Business & Finance News" loaded sucessfully!
All data collected!
The next button is not clickable. We have navigated through all the pages!


In [60]:
len(data)

314

In [142]:
stock_df = pd.DataFrame(data)

In [143]:
stock_df

,name,symbol,price,change,volume,market_cap,pe_ratio
0,NVIDIA Corporation,NVDA,167.52,-3.72,167.475M,4.072T,34.19
1,Grab Holdings Limited,GRAB,3.57,-0.14,66.352M,14.639B,59.50
2,Intel Corporation,INTC,43.13,-0.97,60.753M,216.556B,--
3,"Tesla, Inc.",TSLA,361.83,-10.28,57.765M,1.358T,335.03
4,Nu Holdings Ltd.,NU,13.60,-0.42,56.269M,66.037B,23.26
...,...,...,...,...,...,...,...
309,"Fidelity National Information Services, Inc.",FIS,46.89,-0.52,5.142M,24.282B,64.23
310,"Host Hotels & Resorts, Inc.",HST,18.84,-0.58,5.125M,13.126B,17.13
311,Almonty Industries Inc.,ALM,14.91,+0.23,5.046M,4.204B,--
312,Huntsman Corporation,HUN,12.66,+0.11,5.028M,2.203B,--


In [144]:
# removing the trailing and leading spaces of only the object columns

for col in stock_df.columns:
    if stock_df[col].dtype == 'object':
        col.str.strip()

In [145]:
# renaming the price column
stock_df.rename(columns={'price':'price(USD)','volume':'volume(M)'},inplace=True)

In [146]:
stock_df

,name,symbol,price(USD),change,volume(M),market_cap,pe_ratio
0,NVIDIA Corporation,NVDA,167.52,-3.72,167.475M,4.072T,34.19
1,Grab Holdings Limited,GRAB,3.57,-0.14,66.352M,14.639B,59.50
2,Intel Corporation,INTC,43.13,-0.97,60.753M,216.556B,--
3,"Tesla, Inc.",TSLA,361.83,-10.28,57.765M,1.358T,335.03
4,Nu Holdings Ltd.,NU,13.60,-0.42,56.269M,66.037B,23.26
...,...,...,...,...,...,...,...
309,"Fidelity National Information Services, Inc.",FIS,46.89,-0.52,5.142M,24.282B,64.23
310,"Host Hotels & Resorts, Inc.",HST,18.84,-0.58,5.125M,13.126B,17.13
311,Almonty Industries Inc.,ALM,14.91,+0.23,5.046M,4.204B,--
312,Huntsman Corporation,HUN,12.66,+0.11,5.028M,2.203B,--


In [147]:
# checking if the pe_ratio column have NAN's or not 
stock_df[stock_df['pe_ratio'].isna()]
    

,name,symbol,price(USD),change,volume(M),market_cap,pe_ratio


In [148]:
stock_df['pe_ratio'].value_counts()

pe_ratio
--       123
48.39      4
85.62      4
22.77      4
16.26      4
        ... 
19.86      1
7.73       1
30.89      1
64.23      1
17.13      1
Name: count, Length: 152, dtype: int64

In [149]:
stock_df['pe_ratio'] = stock_df['pe_ratio'].str.replace('--','0') # now theere data types can be converted
stock_df['pe_ratio'] = stock_df['pe_ratio'].str.replace(',', '')


In [158]:
stock_df['volume(M)'] = stock_df['volume(M)'].str.replace('M','')

In [152]:
stock_df['price(USD)'].isna().sum()  # now there data type can be converted

np.int64(0)

In [153]:
stock_df['change'].value_counts()

change
-0.28    8
-1.96    8
+0.96    6
-0.05    6
-0.64    5
        ..
-2.76    1
+3.81    1
-1.47    1
-0.52    1
-3.84    1
Name: count, Length: 184, dtype: int64

In [154]:
stock_df['change'].isna().sum()

np.int64(0)

In [160]:
for col in stock_df.columns:
    try:
        stock_df[col] = pd.to_numeric(stock_df[col])
    except ValueError:
        # if conversion fails → keep column as it is
        pass

In [162]:
stock_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        314 non-null    str    
 1   symbol      314 non-null    str    
 2   price(USD)  314 non-null    float64
 3   change      314 non-null    float64
 4   volume(M)   314 non-null    float64
 5   market_cap  314 non-null    str    
 6   pe_ratio    314 non-null    float64
dtypes: float64(4), str(3)
memory usage: 17.3 KB


In [161]:
stock_df

,name,symbol,price(USD),change,volume(M),market_cap,pe_ratio
0,NVIDIA Corporation,NVDA,167.52,-3.72,167.475,4.072T,34.19
1,Grab Holdings Limited,GRAB,3.57,-0.14,66.352,14.639B,59.50
2,Intel Corporation,INTC,43.13,-0.97,60.753,216.556B,0.00
3,"Tesla, Inc.",TSLA,361.83,-10.28,57.765,1.358T,335.03
4,Nu Holdings Ltd.,NU,13.60,-0.42,56.269,66.037B,23.26
...,...,...,...,...,...,...,...
309,"Fidelity National Information Services, Inc.",FIS,46.89,-0.52,5.142,24.282B,64.23
310,"Host Hotels & Resorts, Inc.",HST,18.84,-0.58,5.125,13.126B,17.13
311,Almonty Industries Inc.,ALM,14.91,0.23,5.046,4.204B,0.00
312,Huntsman Corporation,HUN,12.66,0.11,5.028,2.203B,0.00


In [163]:
def convert_to_billions(val):
    val = str(val).strip()
    
    if val.endswith('T'):
        return float(val[:-1]) * 1000   # Trillion → Billion
    elif val.endswith('B'):
        return float(val[:-1])          # Already Billion
    else:
        return val  # or keep as is if you want

# apply conversion
stock_df['market_cap(B)'] = stock_df['market_cap'].apply(convert_to_billions)

# drop old column
stock_df.drop(columns=['market_cap'], inplace=True)

In [164]:
stock_df

,name,symbol,price(USD),change,volume(M),pe_ratio,market_cap(B)
0,NVIDIA Corporation,NVDA,167.52,-3.72,167.475,34.19,4072.000
1,Grab Holdings Limited,GRAB,3.57,-0.14,66.352,59.50,14.639
2,Intel Corporation,INTC,43.13,-0.97,60.753,0.00,216.556
3,"Tesla, Inc.",TSLA,361.83,-10.28,57.765,335.03,1358.000
4,Nu Holdings Ltd.,NU,13.60,-0.42,56.269,23.26,66.037
...,...,...,...,...,...,...,...
309,"Fidelity National Information Services, Inc.",FIS,46.89,-0.52,5.142,64.23,24.282
310,"Host Hotels & Resorts, Inc.",HST,18.84,-0.58,5.125,17.13,13.126
311,Almonty Industries Inc.,ALM,14.91,0.23,5.046,0.00,4.204
312,Huntsman Corporation,HUN,12.66,0.11,5.028,0.00,2.203


In [167]:
import pandas as pd
from sqlalchemy import create_engine, text

# 🔹 DB credentials
username = "*****"
password = "*****"
host = "localhost"
port = "3306"
database = "scrap_db"

# 🔹 Step 1: Create connection (without DB first)
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}")

# 🔹 Step 2: Create database if not exists
with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {database}"))
    print(f"Database '{database}' ensured.")

# 🔹 Step 3: Connect to that database
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# 🔹 Step 4: Insert DataFrame (replace table if exists)
table_name = "stocks_data"

stock_df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",   # 🔥 this replaces table if exists
    index=False
)

print(f"Table '{table_name}' created/replaced successfully.")

Database 'scrap_db' ensured.
Table 'stocks_data' created/replaced successfully.
